# 11 · Rejilla de datasets real/sintético

Define la rejilla experimental completa, verifica el montaje de los datasets mixtos y deja la lista de recetas que consume el barrido del notebook 12.

**Entradas**

- `data/processed/ventanas.npz`
- `data/synthetic/*.npz`

**Salidas**

- `results/metricas/rejilla.csv`
- `results/figures/reparto_politicas.png`

**Tiempo estimado:** ~3 min en CPU.

In [ ]:
import sys; sys.path.insert(0, "..")   # permite ejecutar desde notebooks/
import src                              # fija el backend de Keras a PyTorch
from src import config, viz
config.fijar_semillas()
viz.aplicar_estilo()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import ventanas

part = ventanas.cargar_procesado()
train, val, test = part.train, part.val, part.test
print(train, val, test, sep="\n")

In [ ]:
from src import mezclas, regimenes
from src.generadores.base import REGISTRO, GeneradorSintetico

v = config.ventanas()
n_regimenes = config.n_regimenes()

disponibles = [n for n in sorted(REGISTRO) if (src.DIR_SINTETICO / (n + ".npz")).exists()]
faltan = sorted(set(REGISTRO) - set(disponibles))

print("Generadores con banco exportado:", disponibles)
print("Pendientes:", faltan if faltan else "ninguno")

## Los dos ejes del experimento

El enunciado pide *"datasets que tengan distinto porcentaje de datos sintéticos y
reales"*. La rejilla lo materializa sobre dos ejes, no uno:

**Eje 1 — proporción de sintéticos.** Con el conjunto real fijo se añaden 0, 0.5×,
1×, 2× y 5× muestras sintéticas.

**Eje 2 — escasez de datos reales.** Todo lo anterior se repite partiendo de 250,
500, 1.000, 2.000 y todas las ventanas reales.

El segundo eje es imprescindible. El beneficio de los sintéticos se concentra en el
régimen de pocos datos y se desvanece —o se vuelve negativo— cuando ya hay muchos
reales. Barrer solo el porcentaje de sintéticos con todos los reales disponibles
produciría una línea plana y la conclusión errónea de que los generadores no sirven.

Las combinaciones con ratio 0 no dependen del generador ni de la política: se emiten
una sola vez bajo el nombre `solo_real`. Sin esa deduplicación se entrenaría el mismo
modelo catorce veces por nivel de reales.

In [ ]:
recetas = mezclas.rejilla(disponibles)
resumen = mezclas.resumen_rejilla(recetas)

print("Recetas:", len(recetas))
print("  de referencia (solo_real):", sum(r.es_referencia for r in recetas))
print("  con sintéticos:          ", sum(not r.es_referencia for r in recetas))
print("Entrenamientos totales (2 tareas):", 2 * len(recetas))
resumen.head(10)

## Políticas de reparto por clase

La hipótesis central del trabajo es que el valor de los sintéticos está en la clase
rara. Se distinguen dos políticas para poder contrastarla:

`proporcional` — el sintético replica el desbalance real. Mide el efecto de *más
datos* sin más.

`equilibrado` — el sintético se concentra en las clases minoritarias hasta
igualarlas. Mide el efecto de *más datos donde hacen falta*.

Si ambas dieran lo mismo, el beneficio sería regularización y no cobertura de la
clase de crisis, y el informe tendría que decirlo.

In [ ]:
base_real = train.submuestra(500)
n_pedidos = 1000

reparto = pd.DataFrame({
    politica: mezclas.repartir(base_real.y_reg, n_pedidos, n_regimenes, politica)
    for politica in mezclas.POLITICAS
})
reparto.index.name = "regimen"
reparto["reales_disponibles"] = np.bincount(base_real.y_reg, minlength=n_regimenes)
reparto

In [ ]:
fig, eje = plt.subplots()

posiciones = np.arange(n_regimenes)
ancho = 0.26
etiquetas = ["calma", "transición", "crisis"][:n_regimenes]

eje.bar(posiciones - ancho, reparto["reales_disponibles"], width=ancho * 0.9,
        color=viz.color("solo_real"), label="reales (500)")
for i, politica in enumerate(mezclas.POLITICAS):
    eje.bar(posiciones + i * ancho, reparto[politica], width=ancho * 0.9,
            color=viz.PALETA[i], label="sintéticos · " + politica)

eje.set_xticks(posiciones)
eje.set_xticklabels(etiquetas)
eje.set_ylabel("ventanas")
eje.set_title("Reparto de 1.000 sintéticos sobre 500 reales")
eje.legend(fontsize=8)
viz.guardar(fig, "reparto_politicas")

## Verificación del montaje

Se monta una receta concreta y se comprueba lo esencial: que el tamaño es el
esperado, que la distribución de clases resultante es la que promete la política y
que las muestras sintéticas llegan con la geometría correcta de `X`.

Las muestras se toman del banco ya exportado, no invocando al generador. Así el
barrido no depende de tener los siete modelos cargados y dos ejecuciones del
notebook 12 usan exactamente las mismas muestras.

In [ ]:
bancos = {n: GeneradorSintetico.importar_muestras(n) for n in disponibles}

ejemplo = mezclas.Receta(generador=disponibles[0], n_reales=500, ratio=2.0, politica="equilibrado")
montado = mezclas.montar(ejemplo, train, bancos[ejemplo.generador], v, n_regimenes)

print(ejemplo.identificador)
print(ejemplo.etiqueta)
print("tamaño esperado:", 500 + int(2.0 * 500), "· obtenido:", len(montado))
print("X:", montado.X.shape, "· esperado:", (len(montado), v.pasado, config.n_canales()))
regimenes.distribucion(montado.y_reg, n_regimenes)

## Lo que no cambia entre recetas

Solo el conjunto de entrenamiento se altera. Validación y test son siempre reales y
siempre los mismos: si cambiaran, las métricas de dos recetas no serían comparables
y todo el análisis quedaría sin sentido.

In [ ]:
print("val :", len(val), "ventanas ·", val.fechas[0].date(), "→", val.fechas[-1].date())
print("test:", len(test), "ventanas ·", test.fechas[0].date(), "→", test.fechas[-1].date())

referencia = mezclas.Receta(generador="solo_real", n_reales=500, ratio=0.0)
solo_real = mezclas.montar(referencia, train, None, v, n_regimenes)
print("receta de referencia:", len(solo_real), "ventanas, todas reales:",
      bool(pd.notna(pd.Series(solo_real.fechas)).all()))

## Rejilla persistida

La lista de recetas se guarda para que el notebook 12 y el informe hablen exactamente
de las mismas combinaciones, y para que cualquier punto de cualquier figura se pueda
rastrear hasta el experimento que lo produjo.

In [ ]:
resumen.to_csv(src.DIR_METRICAS / "rejilla.csv", index=False)
print("Guardadas", len(resumen), "recetas en results/metricas/rejilla.csv")

## Salidas generadas

In [ ]:
from pathlib import Path

salidas = [
    src.DIR_METRICAS / "rejilla.csv",
    src.DIR_FIGURAS / "reparto_politicas.png",
]

for ruta in salidas:
    ruta = Path(ruta)
    marca = "ok" if ruta.exists() else "--"
    print("[{}] {}".format(marca, ruta.relative_to(src.RAIZ)))
